# G20 - DeepSeek-R1-Distill-Qwen-7B (`deepseek-ai/DeepSeek-R1-Distill-Qwen-7B`)
**M.Tech Laboratory CA - Pretrained Models.**  Paper: *DeepSeek-R1: Incentivizing Reasoning Capability in LLMs via Reinforcement Learning* (DeepSeek-AI, 2025) - https://arxiv.org/abs/2501.12948 ·
Official repo: https://github.com/deepseek-ai/DeepSeek-R1 · Model card: https://huggingface.co/deepseek-ai/DeepSeek-R1-Distill-Qwen-7B

**Important distinction.** The paper's RL pipeline (R1-Zero: pure RL with GRPO; R1: cold-start SFT -> reasoning RL -> rejection-sampling SFT -> all-scenario RL) produced the 671B-parameter MoE **DeepSeek-R1**.
The checkpoint used here is a **dense 7B model obtained by supervised fine-tuning (distillation only, no RL stage)** of the base model **Qwen2.5-Math-7B** on ~800k samples generated by DeepSeek-R1
(paper §2.4; repo README table). It did **not** itself undergo the R1 RL training.

| Item | Value (source) |
|---|---|
| Exact checkpoint | `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` (never replaced) |
| Base model | Qwen2.5-Math-7B (repo README) |
| Parameters | 7B class (HF reports ~7.6B; counted at load time below); weights 8.61 GB + 6.62 GB safetensors (HF file listing) |
| Licence | MIT (repo + weights); derived from Qwen2.5 (Apache 2.0) |
| Context | read from `config.json` at load time (printed below); evaluation in the paper used up to 32,768 generated tokens |
| Recommended use (repo README) | temperature 0.5-0.7 (0.6), **no system prompt**, put all instructions in the user turn, ask to reason step by step, **force the response to start with `<think>\n`** |
| Paper-reported (NOT ours) | AIME24 pass@1 55.5, MATH-500 92.8, GPQA-Diamond 49.1, LiveCodeBench 37.6, Codeforces 1189 (Table 5; sampling T=0.6, top-p 0.95, up to 64 samples/question, 32k tokens) |

## How to run (Google Colab)
1. **Runtime -> Change runtime type -> T4 GPU** (or better). 2. Run all cells top-to-bottom (`Runtime -> Run all`).
3. No Hugging Face token is required for `deepseek-ai/DeepSeek-R1-Distill-Qwen-7B` (public, MIT). A token, if provided as secret `HF_TOKEN`, only raises download rate limits.
4. At the end the notebook saves `G20_results.json`, prediction files, CSVs and charts into `/content/Pretrained_Model_CA/G20_DeepSeek/results/`
   and downloads a ZIP. Put those files into the matching `results/` folder of the project; the comparison notebook reads them.

> **Honesty note.** Nothing in this notebook is pre-filled. Every number is produced by *your* run. Where an experiment cannot run on your hardware
> the notebook records `SKIPPED`/`FAILED` with the reason instead of inventing a value.
> These are **pretrained** models: no weights are trained here. The dataset is an *evaluation* set only.

## 1. Installation and dependency setup

In [ ]:
# Colab already ships torch / numpy / pandas / matplotlib. We only add / upgrade what we need.
# If pip reports a dependency conflict with a Colab preinstalled package, it is usually harmless; if imports fail, Runtime -> Restart session and re-run.
!pip install -q -U transformers accelerate bitsandbytes huggingface_hub psutil packaging

In [ ]:
#@title Write ca_common.py (embedded copy - run this cell, no need to read it)
_content = r'''"""ca_common.py -- shared evaluation + inference helpers for the Pretrained-Model CA.

Used verbatim by all three notebooks (G19 Gemma, G20 DeepSeek-R1-Distill-Qwen-7B,
G19-vs-G20 comparison).  The pure-python part (metrics, extraction, evaluators,
aggregation) has no heavy dependencies and is unit-tested outside Colab.
The HF/torch part (model loading, generation, memory measurement) imports torch
lazily and can only run where a GPU/CPU torch install exists (Colab).

NOTE ON METRICS: ROUGE here is a small self-contained implementation
(lower-cased word tokens, no stemming).  Absolute values therefore differ
slightly from the `rouge-score` package; it is applied identically to both models.
"""
import json, re, time, math, statistics, string, subprocess, sys, tempfile, os, gc, csv, io

# ----------------------------------------------------------------------------
# 1. Text normalisation and overlap metrics
# ----------------------------------------------------------------------------
_PUNCT = set(string.punctuation) | {"\u2018", "\u2019", "\u201c", "\u201d", "\u20b9", "\u2013", "\u2014"}


def normalize(s):
    """SQuAD-style normalisation: lower-case, strip punctuation/articles/extra spaces."""
    s = (s or "").lower()
    s = "".join(ch for ch in s if ch not in _PUNCT)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def words(s):
    return normalize(s).split()


def exact_match(pred, ref):
    return float(normalize(pred) == normalize(ref))


def token_f1(pred, ref):
    p, r = words(pred), words(ref)
    if not p or not r:
        return float(p == r)
    common = {}
    for w in p:
        if w in r:
            common[w] = min(p.count(w), r.count(w))
    overlap = sum(common.values())
    if overlap == 0:
        return 0.0
    prec, rec = overlap / len(p), overlap / len(r)
    return 2 * prec * rec / (prec + rec)


def _ngrams(toks, n):
    return [tuple(toks[i:i + n]) for i in range(len(toks) - n + 1)]


def rouge_n(pred, ref, n=1):
    p, r = _ngrams(words(pred), n), _ngrams(words(ref), n)
    if not p or not r:
        return {"p": 0.0, "r": 0.0, "f": 0.0}
    from collections import Counter
    cp, cr = Counter(p), Counter(r)
    overlap = sum((cp & cr).values())
    prec, rec = overlap / len(p), overlap / len(r)
    f = 0.0 if overlap == 0 else 2 * prec * rec / (prec + rec)
    return {"p": prec, "r": rec, "f": f}


def _lcs(a, b):
    if not a or not b:
        return 0
    prev = [0] * (len(b) + 1)
    for x in a:
        cur = [0]
        for j, y in enumerate(b, 1):
            cur.append(prev[j - 1] + 1 if x == y else max(prev[j], cur[-1]))
        prev = cur
    return prev[-1]


def rouge_l(pred, ref):
    p, r = words(pred), words(ref)
    if not p or not r:
        return {"p": 0.0, "r": 0.0, "f": 0.0}
    l = _lcs(p, r)
    prec, rec = l / len(p), l / len(r)
    f = 0.0 if l == 0 else 2 * prec * rec / (prec + rec)
    return {"p": prec, "r": rec, "f": f}


def wilson_interval(k, n, z=1.96):
    """95% Wilson score interval for a proportion k/n."""
    if n == 0:
        return (0.0, 0.0)
    ph = k / n
    den = 1 + z * z / n
    centre = (ph + z * z / (2 * n)) / den
    half = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / den
    return (max(0.0, centre - half), min(1.0, centre + half))


# ----------------------------------------------------------------------------
# 2. Output post-processing (reasoning trace vs final answer)
# ----------------------------------------------------------------------------
SPECIAL_KEEP = ("<think>", "</think>")
SPECIAL_EXTRA = ("<end_of_turn>", "<start_of_turn>", "<eos>", "<bos>", "<pad>",
                 "<\uff5cend\u2581of\u2581sentence\uff5c>", "<\uff5cbegin\u2581of\u2581sentence\uff5c>")


def clean_special_tokens(text, special_tokens=()):
    for st in list(special_tokens) + list(SPECIAL_EXTRA):
        if st in SPECIAL_KEEP:
            continue
        text = text.replace(st, "")
    return text.strip()


def split_reasoning(text, expects_think=False):
    """Return dict(reasoning, final, think_closed).

    expects_think=True (DeepSeek-R1 family): the chat template (or our forced prefix)
    already opened <think>, so the generated text normally contains only '</think>'.
    If no '</think>' was generated, the model never finished thinking (usually the
    token budget ran out): final answer is empty and think_closed=False.
    """
    text = text or ""
    if "</think>" in text:
        head, tail = text.split("</think>", 1)
        return {"reasoning": head.replace("<think>", "").strip(), "final": tail.strip(), "think_closed": True}
    if expects_think:
        return {"reasoning": text.replace("<think>", "").strip(), "final": "", "think_closed": False}
    return {"reasoning": "", "final": text.strip(), "think_closed": None}


# ----------------------------------------------------------------------------
# 3. Answer extraction
# ----------------------------------------------------------------------------
_NUM = re.compile(r"-?\d[\d,]*\.?\d*|-?\.\d+")


def _to_float(tok):
    tok = tok.strip().strip(",.")
    tok = tok.replace(",", "")
    try:
        return float(tok)
    except ValueError:
        return None


def extract_number(final):
    """Priority: 'Final answer: ..' line  >  \\boxed{..}  >  last number in text."""
    if not final:
        return None
    m = re.findall(r"final\s+answer\s*[:\-\u2013]?\s*(.*)", final, flags=re.I)
    cands = []
    if m:
        cands.append(m[-1])
    b = re.findall(r"\\boxed\{([^{}]*)\}", final)
    if b:
        cands.append(b[-1])
    for c in cands:
        nums = _NUM.findall(c)
        if nums:
            v = _to_float(nums[0])
            if v is not None:
                return v
    nums = _NUM.findall(final)
    for tok in reversed(nums):
        v = _to_float(tok)
        if v is not None:
            return v
    return None


def extract_yes_no(final):
    head = (final or "")[:120].lower()
    m = re.search(r"\b(yes|no)\b", head)
    return m.group(1) if m else None


def fmt_num(v):
    if v is None:
        return ""
    return str(int(v)) if float(v).is_integer() else ("%g" % v)


# ----------------------------------------------------------------------------
# 4. Constraint / format checkers
# ----------------------------------------------------------------------------
def count_sentences(text):
    parts = [p for p in re.split(r"(?<=[.!?])\s+", text.strip()) if p.strip()]
    return len(parts)


def check_constraints(text, spec):
    """Generic instruction-following checker.  Returns dict(check -> bool)."""
    out = {}
    t = (text or "").strip()
    lines = [l for l in t.splitlines() if l.strip()]
    if "n_lines" in spec:
        out["n_lines"] = len(lines) == spec["n_lines"]
    if spec.get("no_digits"):
        out["no_digits"] = not re.search(r"\d", t)
    if spec.get("no_bullets"):
        out["no_bullets"] = all(not re.match(r"\s*([-*\u2022]|\d+[.)])\s", l) for l in lines) and bool(lines)
    if "line_max_words" in spec:
        out["line_max_words"] = bool(lines) and all(len(l.split()) <= spec["line_max_words"] for l in lines)
    if "n_sentences" in spec:
        out["n_sentences"] = count_sentences(t) == spec["n_sentences"]
    if "max_words" in spec:
        out["max_words"] = 0 < len(t.split()) <= spec["max_words"]
    if "min_words" in spec:
        out["min_words"] = len(t.split()) >= spec["min_words"]
    for w in spec.get("forbidden_words", []):
        out["no_word_" + w] = not re.search(r"\b" + re.escape(w) + r"\b", t, flags=re.I)
    for w in spec.get("forbidden_substrings", []):
        out["no_substr_" + w] = w.lower() not in t.lower()
    return out


def keyword_groups(text, groups):
    """Each group is a list of alternatives; every group needs >=1 alternative present (prefix match)."""
    low = (text or "").lower()
    hits = []
    for g in groups:
        hits.append(any(re.search(r"\b" + re.escape(a.lower()), low) for a in g))
    return hits


def _strip_fences(text):
    t = (text or "").strip()
    m = re.match(r"^```[a-zA-Z0-9_+-]*\s*\n(.*?)\n?```\s*$", t, flags=re.S)
    return m.group(1).strip() if m else t


def _find_fenced(text):
    m = re.search(r"```[a-zA-Z0-9_+-]*\s*\n(.*?)```", text or "", flags=re.S)
    return m.group(1).strip() if m else None


def check_structured(text, fmt, spec):
    """fmt in json_object | json_array | csv | md_table | key_value.
    Returns dict(valid, schema_ok, values_ok, clean)."""
    raw = (text or "").strip()
    body = _strip_fences(raw)
    res = {"valid": False, "schema_ok": False, "values_ok": False, "clean": raw == body or raw.startswith("```")}
    if fmt in ("json_object", "json_array"):
        obj = None
        try:
            obj = json.loads(body)
        except Exception:
            fenced = _find_fenced(raw)
            if fenced:
                try:
                    obj = json.loads(fenced)
                    res["clean"] = False
                except Exception:
                    obj = None
        if obj is None:
            return res
        res["valid"] = True
        if fmt == "json_object" and isinstance(obj, dict):
            res["schema_ok"] = set(spec["keys"]).issubset(obj.keys())
            res["values_ok"] = res["schema_ok"] and all(
                str(obj.get(k)).strip().lower() == str(v).strip().lower() for k, v in spec["values"].items())
        elif fmt == "json_array" and isinstance(obj, list):
            n = spec.get("len")
            ok = (n is None or len(obj) == n) and all(isinstance(o, dict) and set(spec["keys"]).issubset(o.keys()) for o in obj)
            res["schema_ok"] = ok
            vs = spec.get("values_seq")
            if ok and vs:
                res["values_ok"] = len(obj) == len(vs) and all(
                    all(str(o.get(k)).strip().lower() == str(v).strip().lower() for k, v in e.items())
                    for o, e in zip(obj, vs))
            elif ok:
                res["values_ok"] = True
        return res
    if fmt == "csv":
        try:
            rows = list(csv.reader(io.StringIO(body)))
            rows = [r for r in rows if any(c.strip() for c in r)]
        except Exception:
            return res
        res["valid"] = len(rows) >= 2 and len({len(r) for r in rows}) == 1
        if res["valid"]:
            hdr = [c.strip().lower() for c in rows[0]]
            res["schema_ok"] = hdr == [c.lower() for c in spec["columns"]]
            exp = [[str(c).lower() for c in r] for r in spec["rows"]]
            got = [[c.strip().lower() for c in r] for r in rows[1:]]
            res["values_ok"] = res["schema_ok"] and got == exp
        return res
    if fmt == "md_table":
        tl = [l.strip() for l in body.splitlines() if l.strip().startswith("|")]
        if len(tl) < 3:
            return res
        cells = lambda l: [c.strip() for c in l.strip().strip("|").split("|")]
        sep_ok = all(re.fullmatch(r":?-{2,}:?", c.replace(" ", "")) or re.fullmatch(r":?-+:?", c) for c in cells(tl[1]))
        res["valid"] = sep_ok
        if sep_ok:
            hdr = [c.lower() for c in cells(tl[0])]
            res["schema_ok"] = hdr == [c.lower() for c in spec["columns"]]
            clean = lambda c: re.sub(r"[\u20b9$,\s]", "", c.lower())
            got = [[clean(c) for c in cells(l)] for l in tl[2:]]
            exp = [[clean(str(c)) for c in r] for r in spec["rows"]]
            res["values_ok"] = res["schema_ok"] and got == exp
        res["clean"] = all(l.strip().startswith("|") for l in body.splitlines() if l.strip())
        return res
    if fmt == "key_value":
        pairs = {}
        for l in body.splitlines():
            if ":" in l:
                k, v = l.split(":", 1)
                pairs[k.strip().lower()] = v.strip().lower()
        res["valid"] = len(pairs) >= 1
        res["schema_ok"] = set(k.lower() for k in spec["keys"]).issubset(pairs)
        res["values_ok"] = res["schema_ok"] and all(pairs.get(k.lower()) == str(v).lower() for k, v in spec["values"].items())
        return res
    raise ValueError("unknown fmt " + fmt)


_CLARIFY = re.compile(
    r"clarif|which (one|georgia|text|file|item|unit|language|thing)|do you mean|could you (please )?(specify|provide|tell|share)|"
    r"can you (please )?(specify|provide|tell|share)|what (do you mean|are you referring|text|would you like)|not (sure|clear) (what|which)|"
    r"unclear|ambiguous|more (context|information|details)|need (more|to know)|depends on|refer(s|ring)? to|please (provide|specify|share)|"
    r"you('ve| have) not (said|specified|provided|mentioned)|didn't (specify|provide)|missing", re.I)


def run_code_tests(code, func_name, tests, timeout=10):
    """Run model-generated python in a subprocess (timeout) against [(args, expected), ...].
    SECURITY: executes untrusted generated code -- only run inside Colab / a sandbox."""
    script = (code + "\n\nimport json as _j, sys as _s\n_t = _j.loads(" + repr(json.dumps(tests)) + ")\n_r = []\n"
              "for _a, _e in _t:\n    try:\n        _r.append(bool(" + func_name + "(*_a) == _e))\n"
              "    except Exception:\n        _r.append(False)\n_s.stdout.write('@@RESULT@@' + _j.dumps(_r))\n")
    with tempfile.TemporaryDirectory() as d:
        p = os.path.join(d, "gen_code.py")
        with open(p, "w") as f:
            f.write(script)
        try:
            proc = subprocess.run([sys.executable, p], capture_output=True, text=True, timeout=timeout, cwd=d)
        except subprocess.TimeoutExpired:
            return {"passed": 0, "total": len(tests), "error": "timeout"}
    m = re.search(r"@@RESULT@@(.*)$", proc.stdout, flags=re.S)
    if not m:
        return {"passed": 0, "total": len(tests), "error": (proc.stderr or "no output")[-300:]}
    r = json.loads(m.group(1))
    return {"passed": sum(r), "total": len(r), "error": None}


def extract_code(text):
    m = re.findall(r"```(?:python|py)?\s*\n(.*?)```", text or "", flags=re.S)
    if m:
        return max(m, key=len).strip()
    return (text or "").strip()


# ----------------------------------------------------------------------------
# 5. Per-sample evaluation
# ----------------------------------------------------------------------------
def evaluate_sample(sample, final_text):
    """Return a dict of per-sample evaluation results (see aggregate() for how they are used)."""
    et, spec = sample["eval_type"], sample.get("eval_spec", {})
    ev = {"correct": False, "extracted": "", "em": None, "f1": None, "compliance": None,
          "structured_valid": None, "rouge": None, "details": {}}
    ref_short = sample.get("short_answer")
    final_text = final_text or ""
    if et == "numeric":
        v = extract_number(final_text)
        ev["extracted"] = fmt_num(v)
        answers = spec.get("answers", [spec.get("answer")])
        tol = spec.get("tol", 0.01)
        ev["correct"] = v is not None and any(abs(v - a) <= tol for a in answers)
        ev["em"] = ev["f1"] = float(ev["correct"])
    elif et == "yesno":
        v = extract_yes_no(final_text)
        ev["extracted"] = v or ""
        ev["correct"] = v == spec["answer"]
        ev["em"] = ev["f1"] = float(ev["correct"])
    elif et == "contains_any":
        low = normalize(final_text)
        ev["correct"] = any(re.search(r"\b" + re.escape(normalize(a)) + r"\b", low) for a in spec["answers"])
        ev["extracted"] = final_text.strip()[:200]
        ev["em"] = max(exact_match(final_text, a) for a in spec["answers"])
        ev["f1"] = max(token_f1(final_text, a) for a in spec["answers"])
    elif et == "keywords":
        hits = keyword_groups(final_text, spec["groups"])
        ev["details"]["group_hits"] = hits
        ev["correct"] = all(hits)
        ev["extracted"] = final_text.strip()[:200]
    elif et == "ifcheck":
        chk = check_constraints(final_text, spec)
        ev["details"]["checks"] = chk
        ev["correct"] = bool(chk) and all(chk.values())
        ev["compliance"] = ev["correct"]
        ev["extracted"] = final_text.strip()[:200]
    elif et == "summary":
        chk = check_constraints(final_text, {"max_words": spec["max_words"]})
        hits = keyword_groups(final_text, spec["groups"])
        ref = sample["reference"]
        ev["rouge"] = {"rouge1": rouge_n(final_text, ref, 1)["f"], "rouge2": rouge_n(final_text, ref, 2)["f"],
                       "rougeL": rouge_l(final_text, ref)["f"]}
        ev["details"] = {"checks": chk, "group_hits": hits}
        ev["compliance"] = all(chk.values())
        ev["correct"] = ev["compliance"] and all(hits)
        ev["extracted"] = final_text.strip()[:300]
    elif et == "code":
        code = extract_code(final_text)
        r = run_code_tests(code, spec["func"], spec["tests"])
        ev["details"] = r
        ev["correct"] = r["total"] > 0 and r["passed"] == r["total"]
        ev["extracted"] = code[:300]
    elif et == "structured":
        r = check_structured(final_text, spec["fmt"], spec["spec"])
        ev["details"] = r
        ev["structured_valid"] = bool(r["valid"])
        ev["correct"] = bool(r["valid"] and r["schema_ok"] and r["values_ok"])
        ev["compliance"] = bool(r["clean"] and r["valid"])  # "output only the structure"
        ev["extracted"] = final_text.strip()[:300]
    elif et == "ambiguity":
        clar = bool(_CLARIFY.search(final_text))
        terms = spec.get("all_terms")
        both = bool(terms) and all(t.lower() in final_text.lower() for t in terms)
        ev["details"] = {"asks_or_flags_ambiguity": clar, "covers_all_readings": both}
        ev["correct"] = clar or both
        ev["extracted"] = final_text.strip()[:200]
    else:
        raise ValueError("unknown eval_type " + et)
    if ref_short and ev["em"] is None:
        ev["em"] = exact_match(ev["extracted"], ref_short)
        ev["f1"] = token_f1(ev["extracted"], ref_short)
    ev["reasoning_correct"] = ev["correct"] if sample.get("reasoning") else None
    return ev


# ----------------------------------------------------------------------------
# 6. Running + aggregation
# ----------------------------------------------------------------------------
def run_samples(samples, generate_fn, expects_think=False, log=print, tag=""):
    """generate_fn(prompt) -> dict(text, input_tokens, new_tokens, seconds, hit_max_tokens, mem=dict(...))."""
    records = []
    for i, s in enumerate(samples, 1):
        g = generate_fn(s["prompt"])
        parts = split_reasoning(g["text"], expects_think=expects_think)
        ev = evaluate_sample(s, parts["final"])
        rec = {k: s.get(k) for k in ("id", "split", "category", "difficulty", "prompt", "reference", "criteria")}
        rec.update({
            "variation": s.get("variation"), "expected_behaviour": s.get("expected_behaviour"),
            "raw_output": g["text"], "final_output": parts["final"], "reasoning_chars": len(parts["reasoning"]),
            "think_closed": parts["think_closed"], "input_tokens": g["input_tokens"], "new_tokens": g["new_tokens"],
            "seconds": g["seconds"], "tokens_per_sec": g["new_tokens"] / g["seconds"] if g["seconds"] > 0 else None,
            "hit_max_tokens": g["hit_max_tokens"], "mem": g.get("mem"), "eval": ev,
        })
        rec["observation"] = auto_observation(rec)
        records.append(rec)
        log("[%s %d/%d] %s %-5s %.1fs %d tok  %s" % (tag, i, len(samples), s["id"], "PASS" if ev["correct"] else "FAIL",
                                                      g["seconds"], g["new_tokens"], (parts["final"] or g["text"])[:60].replace("\n", " ")))
    return records


def auto_observation(rec):
    ev = rec["eval"]
    bits = ["PASS" if ev["correct"] else "FAIL", "%d new tokens" % rec["new_tokens"], "%.1fs" % rec["seconds"]]
    if rec.get("hit_max_tokens"):
        bits.append("stopped at max_new_tokens limit")
    if rec.get("think_closed") is False:
        bits.append("reasoning trace never closed (no </think>) -> no final answer")
    if not ev["correct"] and ev.get("extracted"):
        bits.append("extracted=%r" % ev["extracted"][:60])
    return "; ".join(bits)


def _mean(xs):
    xs = [x for x in xs if x is not None]
    return sum(xs) / len(xs) if xs else None


def aggregate(records):
    n = len(records)
    k = sum(1 for r in records if r["eval"]["correct"])
    lo, hi = wilson_interval(k, n)
    secs = [r["seconds"] for r in records]
    tps = [r["tokens_per_sec"] for r in records if r["tokens_per_sec"]]
    cats = {}
    for r in records:
        c = cats.setdefault(r["category"], {"n": 0, "correct": 0})
        c["n"] += 1
        c["correct"] += int(r["eval"]["correct"])
    for c in cats.values():
        c["accuracy"] = c["correct"] / c["n"]
    rs = [r["eval"]["rouge"] for r in records if r["eval"]["rouge"]]
    reasoning = [r["eval"]["reasoning_correct"] for r in records if r["eval"]["reasoning_correct"] is not None]
    comp = [r["eval"]["compliance"] for r in records if r["eval"]["compliance"] is not None]
    sv = [r["eval"]["structured_valid"] for r in records if r["eval"]["structured_valid"] is not None]
    tot_new, tot_sec = sum(r["new_tokens"] for r in records), sum(secs)
    peaks = [r["mem"]["peak_allocated_gb"] for r in records if r.get("mem") and r["mem"].get("peak_allocated_gb") is not None]
    return {
        "n": n, "n_correct": k, "accuracy": k / n if n else None, "accuracy_ci95": [lo, hi],
        "exact_match": _mean([r["eval"]["em"] for r in records]),
        "token_f1": _mean([r["eval"]["f1"] for r in records]),
        "reasoning_correctness": _mean([float(x) for x in reasoning]), "n_reasoning": len(reasoning),
        "instruction_compliance": _mean([float(x) for x in comp]), "n_compliance": len(comp),
        "structured_validity": _mean([float(x) for x in sv]), "n_structured": len(sv),
        "rouge1": _mean([x["rouge1"] for x in rs]), "rouge2": _mean([x["rouge2"] for x in rs]),
        "rougeL": _mean([x["rougeL"] for x in rs]), "n_rouge": len(rs),
        "per_category": cats,
        "time_mean_s": _mean(secs), "time_median_s": statistics.median(secs) if secs else None,
        "time_p90_s": sorted(secs)[int(0.9 * (len(secs) - 1))] if secs else None, "time_total_s": tot_sec,
        "tokens_per_sec_aggregate": tot_new / tot_sec if tot_sec > 0 else None,
        "tokens_per_sec_median": statistics.median(tps) if tps else None,
        "mean_new_tokens": tot_new / n if n else None,
        "peak_gpu_mem_gb_max": max(peaks) if peaks else None,
        "n_hit_max_tokens": sum(1 for r in records if r.get("hit_max_tokens")),
        "n_unfinished_thinking": sum(1 for r in records if r.get("think_closed") is False),
    }


_FAIL_TYPE = {
    "numeric": "Mathematical / multi-step reasoning error", "yesno": "Logical reasoning error",
    "contains_any": "Factual inaccuracy (possible hallucination)", "keywords": "Analytical / content-coverage failure",
    "ifcheck": "Instruction-following failure", "summary": "Summarisation constraint or coverage failure",
    "code": "Coding error", "structured": "Incorrect structured output", "ambiguity": "Ambiguous-prompt mis-handling",
}


def build_failure_table(records_with_types):
    """records_with_types: list of (record, eval_type). Returns list of failure dicts (observed failures only)."""
    out = []
    for rec, et in records_with_types:
        ev = rec["eval"]
        if ev["correct"]:
            continue
        ftype = None
        if rec.get("think_closed") is False:
            ftype = "Excessive response / truncation (reasoning never finished within token budget)"
        elif rec.get("hit_max_tokens"):
            ftype = "Excessive response / truncation (hit max_new_tokens)"
        else:
            ftype = _FAIL_TYPE.get(et, "Other")
        out.append({
            "id": rec["id"], "category": rec["category"], "input": rec["prompt"],
            "expected_behaviour": rec.get("expected_behaviour") or ("reference: %s" % rec.get("reference")),
            "actual_output": (rec["final_output"] or rec["raw_output"])[:800], "failure_type": ftype,
            "explanation": "Automatic evaluator marked this FAIL (%s). Extracted=%r. %s" % (
                json.dumps(ev.get("details", {}), default=str)[:240], ev.get("extracted", "")[:80],
                "Manual review recommended before citing in the report."),
            "observed": True,
        })
    return out


def consistency_test(generate_fn, sample, seeds=(0, 1, 2), expects_think=False):
    outs = []
    for sd in seeds:
        g = generate_fn(sample["prompt"], seed=sd)
        parts = split_reasoning(g["text"], expects_think=expects_think)
        ev = evaluate_sample(sample, parts["final"])
        outs.append({"seed": sd, "extracted": ev["extracted"], "correct": ev["correct"], "new_tokens": g["new_tokens"]})
    answers = {o["extracted"] for o in outs}
    return {"id": sample["id"], "runs": outs, "n_distinct_answers": len(answers), "all_correct": all(o["correct"] for o in outs)}


def save_json(path, obj):
    os.makedirs(os.path.dirname(os.path.abspath(path)), exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False, default=str)
    return path


def load_json(path):
    with open(path, encoding="utf-8") as f:
        return json.load(f)


# ----------------------------------------------------------------------------
# 7. HF / torch runtime (Colab only)
# ----------------------------------------------------------------------------
def env_report():
    import platform
    info = {"python": platform.python_version(), "platform": platform.platform()}
    try:
        import torch
        info["torch"] = torch.__version__
        info["cuda_available"] = torch.cuda.is_available()
        info["cuda_version"] = torch.version.cuda
        if torch.cuda.is_available():
            p = torch.cuda.get_device_properties(0)
            info["gpu"] = p.name
            info["gpu_total_gb"] = round(p.total_memory / 1024 ** 3, 2)
            try:
                info["bf16_native"] = bool(torch.cuda.is_bf16_supported(including_emulation=False))
            except TypeError:
                info["bf16_native"] = bool(torch.cuda.get_device_capability(0)[0] >= 8)
    except Exception as e:  # pragma: no cover
        info["torch_error"] = str(e)
    try:
        import transformers
        info["transformers"] = transformers.__version__
    except Exception:
        pass
    for pkg in ("accelerate", "bitsandbytes", "huggingface_hub", "tokenizers"):
        try:
            info[pkg] = __import__(pkg).__version__
        except Exception:
            info[pkg] = None
    try:
        import psutil
        info["system_ram_gb"] = round(psutil.virtual_memory().total / 1024 ** 3, 2)
    except Exception:
        pass
    return info


def gpu_mem():
    import torch
    if not torch.cuda.is_available():
        return {"allocated_gb": None, "reserved_gb": None, "peak_allocated_gb": None, "peak_reserved_gb": None}
    g = 1024 ** 3
    return {"allocated_gb": torch.cuda.memory_allocated() / g, "reserved_gb": torch.cuda.memory_reserved() / g,
            "peak_allocated_gb": torch.cuda.max_memory_allocated() / g, "peak_reserved_gb": torch.cuda.max_memory_reserved() / g}


def load_hf_model(model_id, precision, token=None, revision=None, log=print):
    """precision in {'bf16','fp16','8bit','4bit'}.  Download time and load time are measured separately."""
    import torch
    import transformers
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from huggingface_hub import snapshot_download
    t0 = time.perf_counter()
    last_err = None
    for attempt in range(1, 4):  # retry flaky downloads
        try:
            snapshot_download(model_id, token=token, revision=revision)
            last_err = None
            break
        except Exception as e:
            last_err = e
            name = type(e).__name__
            if name in ("GatedRepoError", "RepositoryNotFoundError") or "401" in str(e) or "403" in str(e):
                raise  # authentication/licence problems will not fix themselves
            log("download attempt %d failed (%s); retrying in %ds" % (attempt, name, 5 * attempt))
            time.sleep(5 * attempt)
    if last_err is not None:
        raise last_err
    download_s = time.perf_counter() - t0
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
    t1 = time.perf_counter()
    tok = AutoTokenizer.from_pretrained(model_id, token=token, revision=revision)
    tok_s = time.perf_counter() - t1
    kw = {"token": token, "revision": revision, "device_map": {"": 0} if torch.cuda.is_available() else "cpu"}
    try:
        from packaging.version import parse
        dtype_key = "dtype" if parse(transformers.__version__.split("+")[0]) >= parse("4.56.0") else "torch_dtype"
    except Exception:
        dtype_key = "torch_dtype"
    if precision == "bf16":
        kw[dtype_key] = torch.bfloat16
    elif precision == "fp16":
        kw[dtype_key] = torch.float16
    elif precision == "8bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_8bit=True)
        kw[dtype_key] = torch.float16
    elif precision == "4bit":
        kw["quantization_config"] = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                                                       bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
        kw[dtype_key] = torch.float16
    else:
        raise ValueError(precision)
    t2 = time.perf_counter()
    model = AutoModelForCausalLM.from_pretrained(model_id, **kw)
    model.eval()
    if torch.cuda.is_available():
        torch.cuda.synchronize()
    load_s = time.perf_counter() - t2
    mem = gpu_mem()
    n_params = sum(p.numel() for p in model.parameters())
    info = {"model_id": model_id, "revision": revision, "precision": precision, "download_s": download_s,
            "tokenizer_init_s": tok_s, "model_load_s": load_s, "total_load_s": tok_s + load_s, "mem_after_load": mem,
            "n_parameters_loaded": n_params, "max_position_embeddings": getattr(model.config, "max_position_embeddings", None),
            "vocab_size": getattr(model.config, "vocab_size", None)}
    log("loaded %s [%s]: download %.1fs, load %.1fs, GPU alloc %s GB" % (
        model_id, precision, download_s, load_s, None if mem["allocated_gb"] is None else round(mem["allocated_gb"], 2)))
    return model, tok, info


def make_generate_fn(model, tok, gen_kwargs, force_think_prefix=False, default_seed=1234):
    """Returns fn(prompt, seed=None, **gen_overrides) -> dict with timing + memory.
    Uses the tokenizer's own chat template (single user turn, NO system prompt)."""
    import torch
    eos = model.generation_config.eos_token_id
    eos_ids = set(eos if isinstance(eos, (list, tuple)) else [eos]) if eos is not None else set()
    if tok.eos_token_id is not None:
        eos_ids.add(tok.eos_token_id)

    def fn(prompt, seed=None, **over):
        msgs = [{"role": "user", "content": prompt}]
        text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
        if force_think_prefix and not text.rstrip().endswith("<think>"):
            text = text + "<think>\n"
        enc = tok(text, return_tensors="pt", add_special_tokens=False).to(model.device)
        kw = dict(gen_kwargs)
        kw.update(over)
        torch.manual_seed(default_seed if seed is None else seed)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
            torch.cuda.reset_peak_memory_stats()
        t0 = time.perf_counter()
        with torch.no_grad():
            out = model.generate(**enc, pad_token_id=tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id, **kw)
        if torch.cuda.is_available():
            torch.cuda.synchronize()
        dt = time.perf_counter() - t0
        new = out[0, enc["input_ids"].shape[1]:]
        raw = tok.decode(new, skip_special_tokens=False)
        raw = clean_special_tokens(raw, tok.all_special_tokens)
        last = int(new[-1].item()) if new.shape[0] else None
        hit_max = bool(new.shape[0] >= kw.get("max_new_tokens", 10 ** 9) and last not in eos_ids)
        return {"text": raw, "input_tokens": int(enc["input_ids"].shape[1]), "new_tokens": int(new.shape[0]), "seconds": dt,
                "hit_max_tokens": hit_max, "mem": gpu_mem(), "prompt_rendered": text if over.get("_return_prompt") else None}
    return fn


def warmup(gen_fn):
    gen_fn("Say hello.", max_new_tokens=8)


def release(*objs):
    import torch
    for o in objs:
        del o
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def build_long_prompt(tok, target_tokens, needle_code="K7-4921", question="What is the secret access code mentioned in the log? Reply with the code only."):
    """Needle-in-a-haystack style prompt of ~target_tokens tokens (needle placed in the middle)."""
    filler, i = [], 0
    while True:
        i += 1
        filler.append("Log entry %d: station %d recorded %d millimetres of rain and a wind speed of %d km/h." % (i, i % 41 + 1, i % 17 + 2, i % 23 + 5))
        if i % 25 == 0:
            if len(tok("\n".join(filler), add_special_tokens=False)["input_ids"]) >= target_tokens:
                break
    mid = len(filler) // 2
    filler.insert(mid, "NOTE: the secret access code is %s." % needle_code)
    text = "Read the following weather log.\n\n" + "\n".join(filler) + "\n\n" + question
    return text, needle_code


def resource_precision_experiment(model_id, token, precisions, samples, gen_kwargs, expects_think, force_think_prefix, log=print):
    """Load the SAME checkpoint at several precisions; compare load time, memory, speed and quality on `samples`."""
    out = []
    for p in precisions:
        row = {"precision": p, "status": "OK"}
        model = tok = None
        try:
            model, tok, info = load_hf_model(model_id, p, token=token, log=log)
            gen = make_generate_fn(model, tok, gen_kwargs, force_think_prefix=force_think_prefix)
            warmup(gen)
            recs = run_samples(samples, gen, expects_think=expects_think, log=log, tag="res-" + p)
            agg = aggregate(recs)
            row.update({"load": info, "metrics": agg, "peak_mem_gb": agg["peak_gpu_mem_gb_max"],
                        "predictions": [{"id": r["id"], "correct": r["eval"]["correct"], "extracted": r["eval"]["extracted"],
                                         "seconds": r["seconds"], "new_tokens": r["new_tokens"]} for r in recs]})
        except Exception as e:  # OOM, unsupported dtype, missing bitsandbytes ...
            row["status"] = "FAILED: %s: %s" % (type(e).__name__, str(e)[:300])
            log(row["status"])
        finally:
            try:
                release(model, tok)
            except Exception:
                pass
        out.append(row)
    return out


def resource_input_length_experiment(model, tok, gen_fn, lengths, max_ctx=None, expects_think=False, max_new_tokens=None, log=print):
    rows = []
    for L in lengths:
        if max_ctx and L > max_ctx - 600:
            rows.append({"target_tokens": L, "status": "SKIPPED: exceeds context limit %s" % max_ctx})
            continue
        prompt, needle = build_long_prompt(tok, L)
        sample = {"id": "LEN%d" % L, "eval_type": "contains_any", "eval_spec": {"answers": [needle]}, "reference": needle}
        try:
            kw = {"max_new_tokens": max_new_tokens} if max_new_tokens else {}
            g = gen_fn(prompt, **kw)
            parts = split_reasoning(g["text"], expects_think=expects_think)
            ev = evaluate_sample(sample, parts["final"])
            rows.append({"target_tokens": L, "actual_input_tokens": g["input_tokens"], "seconds": g["seconds"], "new_tokens": g["new_tokens"],
                         "peak_mem_gb": g["mem"]["peak_allocated_gb"], "found_needle": ev["correct"], "answer_head": parts["final"][:80],
                         "status": "OK"})
        except Exception as e:
            rows.append({"target_tokens": L, "status": "FAILED: %s: %s" % (type(e).__name__, str(e)[:200])})
        log(rows[-1])
    return rows


def resource_generation_length_experiment(gen_fn, lengths, prompt="Write a detailed explanation of how photosynthesis works, covering light reactions, the Calvin cycle and their locations in the cell.", log=print):
    rows = []
    for L in lengths:
        g = gen_fn(prompt, max_new_tokens=L)
        rows.append({"max_new_tokens": L, "new_tokens": g["new_tokens"], "seconds": g["seconds"],
                     "tokens_per_sec": g["new_tokens"] / g["seconds"], "peak_mem_gb": g["mem"]["peak_allocated_gb"]})
        log(rows[-1])
    return rows
'''
open('ca_common.py', 'w', encoding='utf-8').write(_content)
print('wrote', 'ca_common.py', len(_content), 'chars')


In [ ]:
#@title Write common_eval_dataset.json (embedded copy - run this cell, no need to read it)
_content = r'''{
 "name": "Pretrained_Model_CA common evaluation set",
 "version": "1.0",
 "note": "Evaluation only - pretrained models are NOT trained on this data. dev (20) used to verify the pipeline; test (10) held out, never used to adjust prompts/extractors.",
 "samples": [
  {
   "id": "GK01",
   "split": "dev",
   "category": "general_knowledge",
   "difficulty": "easy",
   "prompt": "What is the chemical symbol for gold? Answer with just the name or number, no explanation.",
   "reference": "Au",
   "eval_type": "contains_any",
   "eval_spec": {
    "answers": [
     "Au"
    ]
   },
   "criteria": "Correct symbol 'Au'",
   "reasoning": false,
   "short_answer": "Au",
   "expected_behaviour": "Reference answer: Au"
  },
  {
   "id": "GK02",
   "split": "dev",
   "category": "general_knowledge",
   "difficulty": "easy",
   "prompt": "What is the capital city of Australia? Answer with just the name or number, no explanation.",
   "reference": "Canberra",
   "eval_type": "contains_any",
   "eval_spec": {
    "answers": [
     "Canberra"
    ]
   },
   "criteria": "Correct capital (common distractors: Sydney, Melbourne)",
   "reasoning": false,
   "short_answer": "Canberra",
   "expected_behaviour": "Reference answer: Canberra"
  },
  {
   "id": "GK03",
   "split": "test",
   "category": "general_knowledge",
   "difficulty": "medium",
   "prompt": "In which year did the Chernobyl nuclear power plant accident occur? Answer with just the name or number, no explanation.",
   "reference": "1986",
   "eval_type": "contains_any",
   "eval_spec": {
    "answers": [
     "1986"
    ]
   },
   "criteria": "Correct year 1986",
   "reasoning": false,
   "short_answer": "1986",
   "expected_behaviour": "Reference answer: 1986"
  },
  {
   "id": "IF01",
   "split": "dev",
   "category": "instruction_following",
   "difficulty": "easy",
   "prompt": "List exactly three fruits, one per line, with no numbering, no bullets and no extra text.",
   "reference": "Three lines, each a fruit name",
   "eval_type": "ifcheck",
   "eval_spec": {
    "n_lines": 3,
    "no_digits": true,
    "no_bullets": true,
    "line_max_words": 3
   },
   "criteria": "Exactly 3 non-empty lines; no digits/bullets; <=3 words per line",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "Exactly three short lines containing fruit names, no numbering/bullets/extra text"
  },
  {
   "id": "IF02",
   "split": "dev",
   "category": "instruction_following",
   "difficulty": "medium",
   "prompt": "Write exactly one sentence about the ocean without using the word 'the'.",
   "reference": "One sentence, no word 'the'",
   "eval_type": "ifcheck",
   "eval_spec": {
    "n_sentences": 1,
    "forbidden_words": [
     "the"
    ]
   },
   "criteria": "Exactly 1 sentence; word 'the' absent",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "A single sentence about the ocean that never uses the word 'the'"
  },
  {
   "id": "IF03",
   "split": "test",
   "category": "instruction_following",
   "difficulty": "medium",
   "prompt": "Write exactly two sentences explaining why sleep is important. Do not use the word 'health' (or any word containing it).",
   "reference": "Two sentences, no 'health'",
   "eval_type": "ifcheck",
   "eval_spec": {
    "n_sentences": 2,
    "forbidden_substrings": [
     "health"
    ]
   },
   "criteria": "Exactly 2 sentences; substring 'health' absent",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "Exactly two sentences about sleep, avoiding 'health'/'healthy'"
  },
  {
   "id": "LR01",
   "split": "dev",
   "category": "logical_reasoning",
   "difficulty": "easy",
   "prompt": "All bloops are razzies. All razzies are lazzies. Are all bloops definitely lazzies? Answer 'Yes' or 'No' first, then explain in one sentence.",
   "reference": "Yes",
   "eval_type": "yesno",
   "eval_spec": {
    "answer": "yes"
   },
   "criteria": "Transitive syllogism -> Yes",
   "reasoning": true,
   "short_answer": "yes",
   "expected_behaviour": "Reference answer: Yes"
  },
  {
   "id": "LR02",
   "split": "dev",
   "category": "logical_reasoning",
   "difficulty": "medium",
   "prompt": "If it rains, the match is cancelled. The match was not cancelled. Can we conclude that it did not rain? Answer 'Yes' or 'No' first, then explain in one sentence.",
   "reference": "Yes",
   "eval_type": "yesno",
   "eval_spec": {
    "answer": "yes"
   },
   "criteria": "Modus tollens -> Yes",
   "reasoning": true,
   "short_answer": "yes",
   "expected_behaviour": "Reference answer: Yes"
  },
  {
   "id": "LR03",
   "split": "test",
   "category": "logical_reasoning",
   "difficulty": "medium",
   "prompt": "All doctors in the clinic are tall. Priya is tall. Is Priya definitely a doctor in the clinic? Answer 'Yes' or 'No' first, then explain in one sentence.",
   "reference": "No",
   "eval_type": "yesno",
   "eval_spec": {
    "answer": "no"
   },
   "criteria": "Affirming the consequent is invalid -> No",
   "reasoning": true,
   "short_answer": "no",
   "expected_behaviour": "Reference answer: No"
  },
  {
   "id": "MR01",
   "split": "dev",
   "category": "mathematical_reasoning",
   "difficulty": "easy",
   "prompt": "What is 15% of 240? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "36",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 36
   },
   "criteria": "Final numeric answer 36",
   "reasoning": true,
   "short_answer": "36",
   "expected_behaviour": "Reference answer: 36"
  },
  {
   "id": "MR02",
   "split": "dev",
   "category": "mathematical_reasoning",
   "difficulty": "easy",
   "prompt": "Solve for x: 3x + 7 = 28. Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "7",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 7
   },
   "criteria": "Final numeric answer 7",
   "reasoning": true,
   "short_answer": "7",
   "expected_behaviour": "Reference answer: 7"
  },
  {
   "id": "MR03",
   "split": "test",
   "category": "mathematical_reasoning",
   "difficulty": "medium",
   "prompt": "A train travels 180 km in 2.5 hours. What is its average speed in km/h? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "72",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 72
   },
   "criteria": "Final numeric answer 72",
   "reasoning": true,
   "short_answer": "72",
   "expected_behaviour": "Reference answer: 72"
  },
  {
   "id": "MS01",
   "split": "dev",
   "category": "multi_step",
   "difficulty": "easy",
   "prompt": "A shop sells pens at 12 rupees each. Riya buys 7 pens and pays with a 100 rupee note. How many rupees of change does she get? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "16",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 16
   },
   "criteria": "7*12=84; 100-84=16",
   "reasoning": true,
   "short_answer": "16",
   "expected_behaviour": "Reference answer: 16"
  },
  {
   "id": "MS02",
   "split": "dev",
   "category": "multi_step",
   "difficulty": "medium",
   "prompt": "Tom is twice as old as Sam. In 6 years the sum of their ages will be 48. How old is Tom now? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "24",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 24
   },
   "criteria": "s + 2s + 12 = 48 -> s=12, Tom=24",
   "reasoning": true,
   "short_answer": "24",
   "expected_behaviour": "Reference answer: 24"
  },
  {
   "id": "MS03",
   "split": "test",
   "category": "multi_step",
   "difficulty": "hard",
   "prompt": "A tank holds 600 litres and is 40% full. A pipe adds 25 litres per minute. How many minutes until the tank is completely full? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "14.4",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 14.4,
    "tol": 0.05
   },
   "criteria": "40% of 600 = 240; remaining 360; 360/25 = 14.4",
   "reasoning": true,
   "short_answer": "14.4",
   "expected_behaviour": "Reference answer: 14.4"
  },
  {
   "id": "SU01",
   "split": "dev",
   "category": "summarization",
   "difficulty": "easy",
   "prompt": "Summarize the following passage in one sentence of at most 25 words.\n\nPassage: Bees visit flowers to collect nectar and pollen for food. As they move from flower to flower, grains of pollen stick to their bodies and are carried to other plants, allowing those plants to produce seeds and fruit. Because many crops depend on this process, a decline in bee populations can reduce harvests and raise food prices.",
   "reference": "Bees pollinate plants while feeding on nectar, so falling bee numbers threaten crop yields and food prices.",
   "eval_type": "summary",
   "eval_spec": {
    "max_words": 25,
    "groups": [
     [
      "bee",
      "bees"
     ],
     [
      "pollinat",
      "pollen"
     ],
     [
      "crop",
      "harvest",
      "food"
     ]
    ]
   },
   "criteria": "<=25 words; mentions bees, pollination, crops/food",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "One concise sentence (<=25 words) covering bees, pollination and crop/food impact"
  },
  {
   "id": "SU02",
   "split": "dev",
   "category": "summarization",
   "difficulty": "medium",
   "prompt": "Summarize the following passage in one sentence of at most 25 words.\n\nPassage: Many companies now allow employees to work from home several days a week. Supporters say this saves commuting time and lets people balance family duties, while critics worry that it weakens team bonds and makes it harder for new staff to learn from colleagues. Surveys of workers often show high satisfaction, but managers remain divided about its long-term effect on productivity.",
   "reference": "Working from home saves time and improves balance, but critics fear weaker teamwork and managers are divided on productivity.",
   "eval_type": "summary",
   "eval_spec": {
    "max_words": 25,
    "groups": [
     [
      "remote",
      "home",
      "hybrid"
     ],
     [
      "saves",
      "flexib",
      "balance",
      "satisf",
      "benefit"
     ],
     [
      "critic",
      "concern",
      "worr",
      "drawback",
      "divided",
      "weak"
     ]
    ]
   },
   "criteria": "<=25 words; covers remote work, benefits and concerns",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "One concise sentence (<=25 words) covering both benefits and concerns of remote work"
  },
  {
   "id": "SU03",
   "split": "test",
   "category": "summarization",
   "difficulty": "medium",
   "prompt": "Summarize the following passage in one sentence of at most 25 words.\n\nPassage: Cities are often several degrees warmer than the countryside around them, an effect called the urban heat island. Dark roads and roofs absorb sunlight, concrete stores heat overnight, and there are fewer trees to provide shade and cool the air through evaporation. Planting trees, installing reflective roofs and adding green spaces can reduce the effect and lower summer energy use.",
   "reference": "Cities run hotter than surroundings because dark surfaces and concrete trap heat, but trees and reflective roofs can cool them.",
   "eval_type": "summary",
   "eval_spec": {
    "max_words": 25,
    "groups": [
     [
      "urban",
      "city",
      "cities"
     ],
     [
      "heat",
      "warm",
      "hot",
      "temperature"
     ],
     [
      "tree",
      "green",
      "reflective",
      "plant",
      "shade",
      "cool"
     ]
    ]
   },
   "criteria": "<=25 words; covers urban heat, cause, mitigation",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "One concise sentence (<=25 words) covering urban heat, its cause and mitigation"
  },
  {
   "id": "CO01",
   "split": "dev",
   "category": "coding",
   "difficulty": "easy",
   "prompt": "Write a Python function `is_palindrome(s)` that returns True if the string s reads the same forwards and backwards when case is ignored, otherwise False. Return only the code in one ```python block.",
   "reference": "def is_palindrome(s): t=s.lower(); return t==t[::-1]",
   "eval_type": "code",
   "eval_spec": {
    "func": "is_palindrome",
    "tests": [
     [
      [
       "Level"
      ],
      true
     ],
     [
      [
       "abc"
      ],
      false
     ],
     [
      [
       ""
      ],
      true
     ],
     [
      [
       "Aa"
      ],
      true
     ],
     [
      [
       "ab"
      ],
      false
     ]
    ]
   },
   "criteria": "Passes 5 unit tests",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "Reference answer: def is_palindrome(s): t=s.lower(); return t==t[::-1]"
  },
  {
   "id": "CO02",
   "split": "dev",
   "category": "coding",
   "difficulty": "easy",
   "prompt": "Write a Python function `sum_even(nums)` that returns the sum of the even integers in the list nums (0 for an empty list). Return only the code in one ```python block.",
   "reference": "def sum_even(nums): return sum(n for n in nums if n%2==0)",
   "eval_type": "code",
   "eval_spec": {
    "func": "sum_even",
    "tests": [
     [
      [
       [
        1,
        2,
        3,
        4
       ]
      ],
      6
     ],
     [
      [
       []
      ],
      0
     ],
     [
      [
       [
        -2,
        5
       ]
      ],
      -2
     ],
     [
      [
       [
        7,
        9
       ]
      ],
      0
     ]
    ]
   },
   "criteria": "Passes 4 unit tests",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "Reference answer: def sum_even(nums): return sum(n for n in nums if n%2==0)"
  },
  {
   "id": "CO03",
   "split": "test",
   "category": "coding",
   "difficulty": "medium",
   "prompt": "Write a Python function `second_largest(nums)` that returns the second largest DISTINCT value in a list of integers, or None if there are fewer than two distinct values. Return only the code in one ```python block.",
   "reference": "def second_largest(nums): d=sorted(set(nums)); return d[-2] if len(d)>1 else None",
   "eval_type": "code",
   "eval_spec": {
    "func": "second_largest",
    "tests": [
     [
      [
       [
        3,
        1,
        2
       ]
      ],
      2
     ],
     [
      [
       [
        5,
        5
       ]
      ],
      null
     ],
     [
      [
       [
        1,
        2,
        2,
        3
       ]
      ],
      2
     ],
     [
      [
       [
        -1,
        -5,
        -3
       ]
      ],
      -3
     ],
     [
      [
       []
      ],
      null
     ]
    ]
   },
   "criteria": "Passes 5 unit tests",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "Reference answer: def second_largest(nums): d=sorted(set(nums)); return d[-2] if len(d)>1 else None"
  },
  {
   "id": "AN01",
   "split": "dev",
   "category": "analytical",
   "difficulty": "medium",
   "prompt": "A store's monthly sales were 120 in January, 150 in February and 135 in March. Which month had the largest month-over-month percentage increase, and what was that percentage?",
   "reference": "February, 25%",
   "eval_type": "keywords",
   "eval_spec": {
    "groups": [
     [
      "february",
      "feb"
     ],
     [
      "25"
     ]
    ]
   },
   "criteria": "Names February and 25%",
   "reasoning": true,
   "short_answer": null,
   "expected_behaviour": "Reference answer: February, 25%"
  },
  {
   "id": "AN02",
   "split": "dev",
   "category": "analytical",
   "difficulty": "medium",
   "prompt": "Plan A costs 500 rupees per month flat. Plan B costs 200 rupees per month plus 10 rupees per GB used. At how many GB of monthly usage do the two plans cost the same? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "30",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 30
   },
   "criteria": "200 + 10x = 500 -> x = 30",
   "reasoning": true,
   "short_answer": "30",
   "expected_behaviour": "Reference answer: 30"
  },
  {
   "id": "AN03",
   "split": "test",
   "category": "analytical",
   "difficulty": "medium",
   "prompt": "Dataset: 4, 8, 6, 5, 3, 9, 100. Is the mean or the median more representative of the typical value, and why? State the median value.",
   "reference": "Median (6) because 100 is an outlier that inflates the mean",
   "eval_type": "keywords",
   "eval_spec": {
    "groups": [
     [
      "median"
     ],
     [
      "6"
     ],
     [
      "outlier",
      "extreme",
      "skew",
      "100"
     ]
    ]
   },
   "criteria": "Chooses median, states 6, justifies with outlier",
   "reasoning": true,
   "short_answer": null,
   "expected_behaviour": "Reference answer: Median (6) because 100 is an outlier that inflates the mean"
  },
  {
   "id": "JS01",
   "split": "dev",
   "category": "structured_output",
   "difficulty": "easy",
   "prompt": "Return a JSON object with the keys \"name\" (string) and \"age\" (integer) for a person named Asha who is 30 years old. Output only the JSON.",
   "reference": "{\"name\": \"Asha\", \"age\": 30}",
   "eval_type": "structured",
   "eval_spec": {
    "fmt": "json_object",
    "spec": {
     "keys": [
      "name",
      "age"
     ],
     "values": {
      "name": "Asha",
      "age": 30
     }
    }
   },
   "criteria": "Valid JSON object, correct keys and values, no extra prose",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "Only a JSON object {\"name\": \"Asha\", \"age\": 30}"
  },
  {
   "id": "JS02",
   "split": "dev",
   "category": "structured_output",
   "difficulty": "medium",
   "prompt": "Return a JSON array of exactly 3 objects, each with the keys \"city\" and \"country\", for Paris, Tokyo and Cairo in that order. Output only the JSON.",
   "reference": "[{\"city\":\"Paris\",\"country\":\"France\"},{\"city\":\"Tokyo\",\"country\":\"Japan\"},{\"city\":\"Cairo\",\"country\":\"Egypt\"}]",
   "eval_type": "structured",
   "eval_spec": {
    "fmt": "json_array",
    "spec": {
     "len": 3,
     "keys": [
      "city",
      "country"
     ],
     "values_seq": [
      {
       "city": "Paris",
       "country": "France"
      },
      {
       "city": "Tokyo",
       "country": "Japan"
      },
      {
       "city": "Cairo",
       "country": "Egypt"
      }
     ]
    }
   },
   "criteria": "Valid JSON array of 3 correct objects in order",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "Only a JSON array of three city/country objects in the requested order"
  },
  {
   "id": "JS03",
   "split": "test",
   "category": "structured_output",
   "difficulty": "medium",
   "prompt": "Produce a Markdown table with the columns Item, Qty, Price for: Pen 10 5; Book 2 120; Bag 1 450. Output only the table.",
   "reference": "| Item | Qty | Price |",
   "eval_type": "structured",
   "eval_spec": {
    "fmt": "md_table",
    "spec": {
     "columns": [
      "Item",
      "Qty",
      "Price"
     ],
     "rows": [
      [
       "Pen",
       "10",
       "5"
      ],
      [
       "Book",
       "2",
       "120"
      ],
      [
       "Bag",
       "1",
       "450"
      ]
     ]
    }
   },
   "criteria": "Valid Markdown table, correct header and 3 correct rows, nothing else",
   "reasoning": false,
   "short_answer": null,
   "expected_behaviour": "Only a Markdown table with header Item|Qty|Price and the three rows"
  },
  {
   "id": "AM01",
   "split": "dev",
   "category": "ambiguous_misleading",
   "difficulty": "medium",
   "prompt": "What is the capital of Georgia?",
   "reference": "Ambiguous: country (Tbilisi) vs US state (Atlanta)",
   "eval_type": "ambiguity",
   "eval_spec": {
    "all_terms": [
     "Tbilisi",
     "Atlanta"
    ]
   },
   "criteria": "Asks which Georgia OR answers for both readings",
   "reasoning": true,
   "short_answer": null,
   "expected_behaviour": "Either asks which Georgia is meant or gives both Tbilisi (country) and Atlanta (US state)"
  },
  {
   "id": "AM02",
   "split": "dev",
   "category": "ambiguous_misleading",
   "difficulty": "medium",
   "prompt": "How many months of the year have 28 days? Think carefully, then give a short answer.",
   "reference": "12 (all months)",
   "eval_type": "keywords",
   "eval_spec": {
    "groups": [
     [
      "12",
      "twelve",
      "all 12",
      "all of them",
      "all months",
      "every month",
      "all twelve"
     ]
    ]
   },
   "criteria": "Answers 12 / all months (trap: February only)",
   "reasoning": true,
   "short_answer": null,
   "expected_behaviour": "Reference answer: 12 (all months)"
  },
  {
   "id": "AM03",
   "split": "test",
   "category": "ambiguous_misleading",
   "difficulty": "hard",
   "prompt": "A bat and a ball cost 1.10 dollars in total. The bat costs 1.00 dollar more than the ball. How much does the ball cost, in cents? Think step by step, then end with a line of the form 'Final answer: <number of cents>'.",
   "reference": "5 cents",
   "eval_type": "numeric",
   "eval_spec": {
    "answers": [
     5,
     0.05
    ],
    "tol": 0.001
   },
   "criteria": "Ball = 5 cents (intuitive wrong answer: 10)",
   "reasoning": true,
   "short_answer": "5",
   "expected_behaviour": "Reference answer: 5 cents"
  }
 ],
 "robustness": [
  {
   "id": "R1a",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V1_prompt_wording",
   "variation_name": "terse + brevity request",
   "prompt": "What is the capital of Australia? Answer with just the city name.",
   "reference": "Canberra",
   "criteria": "Canberra",
   "expected_behaviour": "Canberra",
   "eval_type": "contains_any",
   "eval_spec": {
    "answers": [
     "Canberra"
    ]
   },
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R1b",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V1_prompt_wording",
   "variation_name": "plain paraphrase",
   "prompt": "Name Australia's capital city.",
   "reference": "Canberra",
   "criteria": "Canberra",
   "expected_behaviour": "Canberra",
   "eval_type": "contains_any",
   "eval_spec": {
    "answers": [
     "Canberra"
    ]
   },
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R1c",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V1_prompt_wording",
   "variation_name": "indirect phrasing",
   "prompt": "I forgot: which city serves as the seat of government of Australia? One word please.",
   "reference": "Canberra",
   "criteria": "Canberra",
   "expected_behaviour": "Canberra",
   "eval_type": "contains_any",
   "eval_spec": {
    "answers": [
     "Canberra"
    ]
   },
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R1d",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V1_prompt_wording",
   "variation_name": "telegraphic / lower-case",
   "prompt": "australia capital??",
   "reference": "Canberra",
   "criteria": "Canberra",
   "expected_behaviour": "Canberra",
   "eval_type": "contains_any",
   "eval_spec": {
    "answers": [
     "Canberra"
    ]
   },
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R1e",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V1_prompt_wording",
   "variation_name": "polite verbose",
   "prompt": "Could you kindly tell me, in a full sentence, what the capital of Australia is?",
   "reference": "Canberra",
   "criteria": "Canberra",
   "expected_behaviour": "Canberra",
   "eval_type": "contains_any",
   "eval_spec": {
    "answers": [
     "Canberra"
    ]
   },
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R1f",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V1_prompt_wording",
   "variation_name": "math: symbolic",
   "prompt": "What is 15% of 240? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "36",
   "criteria": "36",
   "expected_behaviour": "36",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 36
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R1g",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V1_prompt_wording",
   "variation_name": "math: words",
   "prompt": "Calculate fifteen percent of two hundred forty. Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "36",
   "criteria": "36",
   "expected_behaviour": "36",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 36
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R1h",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V1_prompt_wording",
   "variation_name": "math: roundabout",
   "prompt": "If you take 240 and find 15 percent of it, what do you get? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "36",
   "criteria": "36",
   "expected_behaviour": "36",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 36
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R2a",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V2_input_complexity",
   "variation_name": "1 step",
   "prompt": "Ravi has 12 apples. He buys 8 more. How many apples does he have? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "20",
   "criteria": "20",
   "expected_behaviour": "20",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 20
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R2b",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V2_input_complexity",
   "variation_name": "2 steps",
   "prompt": "Ravi has 12 apples. He buys 8 more and then gives 5 away. How many apples does he have? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "15",
   "criteria": "15",
   "expected_behaviour": "15",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 15
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R2c",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V2_input_complexity",
   "variation_name": "3 steps",
   "prompt": "Ravi has 12 apples. He buys 8 more, gives 5 away, then triples what he has left. How many apples does he have? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "45",
   "criteria": "45",
   "expected_behaviour": "45",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 45
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R2d",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V2_input_complexity",
   "variation_name": "4 steps",
   "prompt": "Ravi has 12 apples. He buys 8 more, gives 5 away, triples what he has left, eats 5, and finally shares the rest equally among himself and 3 friends (4 people in total). How many apples does each person get? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "10",
   "criteria": "10",
   "expected_behaviour": "10",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 10
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R3a",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V3_ambiguous_instructions",
   "variation_name": "missing referent (edit)",
   "prompt": "Make it shorter.",
   "reference": "Asks what text to shorten (or flags missing input)",
   "criteria": "Asks what text to shorten (or flags missing input)",
   "expected_behaviour": "Asks what text to shorten (or flags missing input)",
   "eval_type": "ambiguity",
   "eval_spec": {},
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R3b",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V3_ambiguous_instructions",
   "variation_name": "pronoun without antecedent",
   "prompt": "Is it safe to eat?",
   "reference": "Asks what 'it' refers to",
   "criteria": "Asks what 'it' refers to",
   "expected_behaviour": "Asks what 'it' refers to",
   "eval_type": "ambiguity",
   "eval_spec": {},
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R3c",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V3_ambiguous_instructions",
   "variation_name": "unspecified units",
   "prompt": "Convert 100 to the right units.",
   "reference": "Asks which quantity/units",
   "criteria": "Asks which quantity/units",
   "expected_behaviour": "Asks which quantity/units",
   "eval_type": "ambiguity",
   "eval_spec": {},
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R3d",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V3_ambiguous_instructions",
   "variation_name": "unspecified language",
   "prompt": "Translate this into the usual language.",
   "reference": "Asks for the text and the target language",
   "criteria": "Asks for the text and the target language",
   "expected_behaviour": "Asks for the text and the target language",
   "eval_type": "ambiguity",
   "eval_spec": {},
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R4a",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V4_math_complexity",
   "variation_name": "level 1: addition",
   "prompt": "What is 48 + 27? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "75",
   "criteria": "75",
   "expected_behaviour": "75",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 75
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R4b",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V4_math_complexity",
   "variation_name": "level 2: 2-digit product",
   "prompt": "What is 17 multiplied by 23? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "391",
   "criteria": "391",
   "expected_behaviour": "391",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 391
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R4c",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V4_math_complexity",
   "variation_name": "level 3: linear equation",
   "prompt": "Solve for x: 5x - 9 = 26. Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "7",
   "criteria": "7",
   "expected_behaviour": "7",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 7
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R4d",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V4_math_complexity",
   "variation_name": "level 4: quadratic roots",
   "prompt": "What is the sum of the roots of x^2 - 9x + 14 = 0? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "9",
   "criteria": "9",
   "expected_behaviour": "9",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 9
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R4e",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V4_math_complexity",
   "variation_name": "level 5: compound interest",
   "prompt": "12000 rupees are invested at 10% interest per year, compounded annually. What is the total amount after 2 years, in rupees? Think step by step, then end with a line of the form 'Final answer: <number>'.",
   "reference": "14520",
   "criteria": "14520",
   "expected_behaviour": "14520",
   "eval_type": "numeric",
   "eval_spec": {
    "answer": 14520,
    "tol": 0.5
   },
   "reasoning": true,
   "short_answer": null
  },
  {
   "id": "R5a",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V5_structured_formats",
   "variation_name": "JSON object",
   "prompt": "Return a JSON object with keys name, age, city for Meera, 28, Pune (age as an integer). Output only the JSON.",
   "reference": "JSON object",
   "criteria": "JSON object",
   "expected_behaviour": "JSON object",
   "eval_type": "structured",
   "eval_spec": {
    "fmt": "json_object",
    "spec": {
     "keys": [
      "name",
      "age",
      "city"
     ],
     "values": {
      "name": "Meera",
      "age": 28,
      "city": "Pune"
     }
    }
   },
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R5b",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V5_structured_formats",
   "variation_name": "JSON array of one object",
   "prompt": "Return a JSON array containing exactly one object with keys name, age, city for Meera, 28, Pune (age as an integer). Output only the JSON.",
   "reference": "JSON array",
   "criteria": "JSON array",
   "expected_behaviour": "JSON array",
   "eval_type": "structured",
   "eval_spec": {
    "fmt": "json_array",
    "spec": {
     "len": 1,
     "keys": [
      "name",
      "age",
      "city"
     ],
     "values_seq": [
      {
       "name": "Meera",
       "age": 28,
       "city": "Pune"
      }
     ]
    }
   },
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R5c",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V5_structured_formats",
   "variation_name": "CSV",
   "prompt": "Return CSV with the header name,age,city and one data row for Meera, 28, Pune. Output only the CSV.",
   "reference": "CSV",
   "criteria": "CSV",
   "expected_behaviour": "CSV",
   "eval_type": "structured",
   "eval_spec": {
    "fmt": "csv",
    "spec": {
     "columns": [
      "name",
      "age",
      "city"
     ],
     "rows": [
      [
       "Meera",
       "28",
       "Pune"
      ]
     ]
    }
   },
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R5d",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V5_structured_formats",
   "variation_name": "Markdown table",
   "prompt": "Return a Markdown table with columns name, age, city and one row for Meera, 28, Pune. Output only the table.",
   "reference": "Markdown table",
   "criteria": "Markdown table",
   "expected_behaviour": "Markdown table",
   "eval_type": "structured",
   "eval_spec": {
    "fmt": "md_table",
    "spec": {
     "columns": [
      "name",
      "age",
      "city"
     ],
     "rows": [
      [
       "Meera",
       "28",
       "Pune"
      ]
     ]
    }
   },
   "reasoning": false,
   "short_answer": null
  },
  {
   "id": "R5e",
   "split": "robustness",
   "category": "robustness",
   "difficulty": "n/a",
   "variation": "V5_structured_formats",
   "variation_name": "key: value lines",
   "prompt": "Return exactly three lines of the form 'key: value' with keys name, age, city for Meera, 28, Pune. Output only those lines.",
   "reference": "key: value lines",
   "criteria": "key: value lines",
   "expected_behaviour": "key: value lines",
   "eval_type": "structured",
   "eval_spec": {
    "fmt": "key_value",
    "spec": {
     "keys": [
      "name",
      "age",
      "city"
     ],
     "values": {
      "name": "Meera",
      "age": 28,
      "city": "Pune"
     }
    }
   },
   "reasoning": false,
   "short_answer": null
  }
 ]
}'''
open('common_eval_dataset.json', 'w', encoding='utf-8').write(_content)
print('wrote', 'common_eval_dataset.json', len(_content), 'chars')


## 2. Environment, GPU and RAM detection (Python / PyTorch / Transformers / CUDA)

In [ ]:
import os, json, time, shutil, importlib, statistics, platform
import ca_common as ca
import torch
ENV = ca.env_report()
for k, v in ENV.items():
    print(f"{k:>18}: {v}")
ALLOW_CPU = False      # set True only for a *very* slow smoke test; results would not be comparable
if not torch.cuda.is_available() and not ALLOW_CPU:
    raise SystemExit("No GPU detected. Runtime -> Change runtime type -> T4 GPU, then re-run. (Nothing was fabricated.)")

## 3. Configuration

In [ ]:
MODEL_KEY, MODEL_ID, PRIMARY_NAME = "G20", "deepseek-ai/DeepSeek-R1-Distill-Qwen-7B", "DeepSeek-R1-Distill-Qwen-7B"
EXPECTS_THINK = True           # generated text = reasoning + '</think>' + answer
FORCE_THINK_PREFIX = True      # README: force the response to start with '<think>\n' (applied only if the template does not already do it)
QUICK_RUN = False
# Paper evaluates with up to 32,768 new tokens. On a Colab T4 that is infeasible (hours/sample), so we cap generation and REPORT how many answers hit the cap.
MAX_NEW_TOKENS = 1536
GEN_KW = dict(do_sample=True, temperature=0.6, top_p=0.95, max_new_tokens=MAX_NEW_TOKENS)   # README-recommended sampling
BASE_DIR = "/content/Pretrained_Model_CA"
RESULTS_DIR = f"{BASE_DIR}/G20_DeepSeek/results"; CHART_DIR = RESULTS_DIR + "/charts"

## 4. Model authentication
The checkpoint is public (MIT) - no licence gate. An optional token avoids anonymous rate limits.

In [ ]:
from huggingface_hub import login
HF_TOKEN = None
try:
    from google.colab import userdata
    HF_TOKEN = userdata.get("HF_TOKEN")
except Exception:
    pass
HF_TOKEN = HF_TOKEN or os.environ.get("HF_TOKEN") or None
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False); print("Logged in with provided token.")
else:
    print("No token provided - using anonymous access (fine for this public model).")

## 5. Exact checkpoint loading + tokenizer initialisation (timed)
A 7B model needs ~15.2 GB in fp16, which does **not** fit a 15 GB T4 together with activations. Policy: if the GPU has >= 22 GB use bf16/fp16 else use **4-bit NF4 (bitsandbytes, fp16 compute)**; if that fails there is no smaller model substitution - the run stops. The precision used is stored in the results and listed as a fairness difference in the comparison.

In [ ]:
gpu_gb = ENV.get("gpu_total_gb") or 0
PRIMARY_PRECISION = ("bf16" if ENV.get("bf16_native") else "fp16") if gpu_gb >= 22 else "4bit"
FALLBACKS = ["8bit", "4bit"] if PRIMARY_PRECISION in ("bf16", "fp16") else []
print("GPU memory (GB):", gpu_gb, "-> primary precision:", PRIMARY_PRECISION)


tried, model, tok = [], None, None
chain = [PRIMARY_PRECISION] + [p for p in FALLBACKS if p != PRIMARY_PRECISION]
for p in chain:
    try:
        model, tok, LOAD_INFO = ca.load_hf_model(MODEL_ID, p, token=HF_TOKEN)
        PRECISION = p
        break
    except torch.cuda.OutOfMemoryError:
        tried.append((p, "CUDA out of memory")); print(f"[{p}] out of memory -> trying next fallback (same checkpoint, lower precision)")
        ca.release()
    except Exception as e:
        tried.append((p, repr(e)[:300])); print(f"[{p}] failed: {e!r}")
        ca.release()
        if any(t in repr(e) for t in ("Gated", "401", "403", "RepositoryNotFound")):
            break   # auth problem: lower precision will not help
if model is None:
    raise RuntimeError(f"Could not load {MODEL_ID}. Attempts: {tried}. The assigned checkpoint is never replaced by another model.")
print("\nLoaded", MODEL_ID, "as", PRECISION, "| attempts before success:", tried)
print("Parameters loaded:", f"{LOAD_INFO['n_parameters_loaded']/1e9:.3f} B", "| max_position_embeddings:", LOAD_INFO["max_position_embeddings"], "| vocab:", LOAD_INFO["vocab_size"])
print(json.dumps({k: v for k, v in LOAD_INFO.items() if k not in ("model_id",)}, indent=1, default=str))

## 6. Tokenizer and prompt formatting
Qwen-style ChatML template from the checkpoint's tokenizer; per the repo README there is **no system prompt**, and the response should start with `<think>\n`. The cell shows what the template actually produces and whether we add the prefix.

In [ ]:
msgs = [{"role": "user", "content": "What is 15% of 240? Please reason step by step."}]
rendered = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
print(repr(rendered)); print("template already ends with '<think>':", rendered.rstrip().endswith("<think>"), "-> forced prefix added by us:", FORCE_THINK_PREFIX and not rendered.rstrip().endswith("<think>"))
ids = tok(rendered, add_special_tokens=False)["input_ids"]
print(len(ids), "tokens:", tok.convert_ids_to_tokens(ids)[:40], "...")
print("vocab size:", len(tok), "| bos/eos/pad:", tok.bos_token, tok.eos_token, tok.pad_token)
print("'<think>' / '</think>' token ids:", tok.convert_tokens_to_ids("<think>"), tok.convert_tokens_to_ids("</think>"))
print("digit tokenisation demo:", tok.tokenize("12345 + 6789"))
print("model generation_config:", model.generation_config)
print(model.config)   # layer count, hidden/intermediate sizes, heads, kv heads, max_position_embeddings

## 7. Inference implementation (reusable, configurable generation parameters)
Same reusable function as G19 (`ca.make_generate_fn`) with README sampling settings. The output is split into the **reasoning trace** (before `</think>`) and the **final answer** (after); *only the final answer is evaluated*. If `</think>` never appears (token budget exhausted while thinking) the answer is empty and the item is counted as a failure of type *excessive response / truncation*.

In [ ]:
gen = ca.make_generate_fn(model, tok, GEN_KW, force_think_prefix=FORCE_THINK_PREFIX)
ca.warmup(gen)            # first CUDA call includes kernel compilation; excluded from all timings
demo = gen("Explain in two sentences what a transformer language model is.")
print(demo["text"]); print({k: demo[k] for k in ("input_tokens", "new_tokens", "seconds", "hit_max_tokens")}); print(demo["mem"])

### Interactive prompt testing
Edit `MY_PROMPTS` or set `INTERACTIVE = True` to type prompts in a loop (blank line to stop). Override generation settings per call, e.g. `gen(p, temperature=0.2, max_new_tokens=100)`.

In [ ]:
MY_PROMPTS = ["How many r's are in the word 'strawberry'? Please reason step by step.", "Give me three tips for learning linear algebra."]
INTERACTIVE = False
for p in MY_PROMPTS:
    g = gen(p); parts = ca.split_reasoning(g["text"], expects_think=True)
    print(">>>", p); print("[reasoning chars]", len(parts["reasoning"]), "| closed:", parts["think_closed"]); print(parts["final"] or "(no final answer - truncated)"); print(f"[{g['new_tokens']} tokens, {g['seconds']:.1f}s]\n")
while INTERACTIVE:
    p = input("prompt> ").strip()
    if not p: break
    g = gen(p); print(g["text"], f"\n[{g['new_tokens']} tokens, {g['seconds']:.1f}s]")

## 8. Common evaluation dataset (30 samples, 20 dev / 10 held-out test)
Identical for G19 and G20. Written by us for this assignment. **Evaluation only** - the pretrained weights are never trained on it.
The dev split was used to verify prompts, answer extraction and evaluators; the **10 held-out test prompts are run once and never used to tune anything**.

In [ ]:
import pandas as pd
DS = ca.load_json("common_eval_dataset.json")
ALL = DS["samples"]; ROB = DS["robustness"]
ET = {s["id"]: s["eval_type"] for s in ALL + ROB}
DEV = [s for s in ALL if s["split"] == "dev"]; TEST = [s for s in ALL if s["split"] == "test"]
if QUICK_RUN:
    DEV, TEST, ROB = DEV[:3], TEST[:2], ROB[:4]; print("QUICK_RUN: reduced sample sets -> results are NOT comparable and are flagged in the JSON")
os.makedirs(RESULTS_DIR, exist_ok=True); os.makedirs(CHART_DIR, exist_ok=True)
pd.set_option("display.max_colwidth", 70, "display.width", 200)
pd.DataFrame(ALL)[["id", "split", "category", "difficulty", "eval_type", "criteria"]]

## 9. Predefined experiments - development split (20)

In [ ]:
dev_recs = ca.run_samples(DEV, gen, expects_think=EXPECTS_THINK, tag="dev")
dev_metrics = ca.aggregate(dev_recs)
ca.save_json(f"{RESULTS_DIR}/{MODEL_KEY}_predictions_dev.json", dev_recs)
pd.DataFrame([{ "id": r["id"], "cat": r["category"], "pass": r["eval"]["correct"], "sec": round(r["seconds"], 2), "new_tok": r["new_tokens"], "extracted": r["eval"]["extracted"][:50]} for r in dev_recs])

## 10. Held-out test split (10) - run once, identical prompts for both models

In [ ]:
test_recs = ca.run_samples(TEST, gen, expects_think=EXPECTS_THINK, tag="test")
test_metrics = ca.aggregate(test_recs)
ca.save_json(f"{RESULTS_DIR}/{MODEL_KEY}_predictions_test.json", test_recs)
pd.DataFrame(test_recs)[["id", "category", "seconds", "new_tokens", "tokens_per_sec", "hit_max_tokens"]].assign(passed=[r["eval"]["correct"] for r in test_recs])

## 11. Evaluation metrics (what they measure, and their limits)
| Metric | Applied to | Measures | Limitation |
|---|---|---|---|
| Accuracy | all samples (task-specific automatic checker) | fraction judged correct | checkers are rule-based; unusual but valid phrasings can be marked wrong |
| Exact Match (EM) | short-answer items | normalised answer equals reference | very strict; verbose answers score 0 |
| Token F1 | short-answer items | word overlap with reference | rewards overlap, not truth |
| ROUGE-1/2/L | summarisation | n-gram/LCS overlap with a single reference | one reference, no semantics (own implementation, no stemming) |
| Reasoning correctness | items flagged `reasoning` (logic, maths, multi-step, analytical, trick questions) | final conclusion correct | does not grade the reasoning chain itself |
| Mathematical answer correctness | numeric items | extracted number within tolerance | extractor uses `Final answer:` > `\\boxed{}` > last number |
| Instruction compliance | IF, summary length, structured-output cleanliness | explicit constraints satisfied | only constraints we can check mechanically |
| Structured-output validity | JSON / table items | parses as the requested format | validity != correctness (checked separately) |
| Response relevance | **not applied** | - | no reliable automatic metric; use the rubric columns for manual review |
The 10-sample test split gives very wide confidence intervals (see `accuracy_ci95`); differences of 1-2 items are **not** statistically meaningful.

In [ ]:
def show(m, title):
    rows = {k: m[k] for k in ("n", "n_correct", "accuracy", "exact_match", "token_f1", "reasoning_correctness", "instruction_compliance",
                              "structured_validity", "rouge1", "rouge2", "rougeL", "time_mean_s", "time_median_s", "tokens_per_sec_aggregate",
                              "peak_gpu_mem_gb_max", "n_hit_max_tokens", "n_unfinished_thinking")}
    rows["accuracy_ci95"] = [round(x, 3) for x in m["accuracy_ci95"]]
    print(title); display(pd.Series(rows, name=title).to_frame())
show(dev_metrics, "DEV (n=%d)" % dev_metrics["n"]); show(test_metrics, "HELD-OUT TEST (n=%d)" % test_metrics["n"])
pd.DataFrame({"dev": {k: v["accuracy"] for k, v in dev_metrics["per_category"].items()}, "test": {k: v["accuracy"] for k, v in test_metrics["per_category"].items()}})

In [ ]:
import matplotlib; import matplotlib.pyplot as plt
both = dev_recs + test_recs
cats = sorted({r["category"] for r in both})
acc = [sum(r["eval"]["correct"] for r in both if r["category"] == c_) / sum(1 for r in both if r["category"] == c_) for c_ in cats]
fig, ax = plt.subplots(1, 2, figsize=(13, 4.2))
ax[0].barh(cats, acc, color="#3b6ea5"); ax[0].set_xlim(0, 1); ax[0].set_title(f"{MODEL_KEY}: accuracy by category (dev+test, 3 items each)"); ax[0].set_xlabel("accuracy")
ax[1].bar([r["id"] for r in test_recs], [r["seconds"] for r in test_recs], color="#a55b3b"); ax[1].set_title(f"{MODEL_KEY}: inference time per held-out test sample"); ax[1].set_ylabel("seconds"); ax[1].tick_params(axis="x", rotation=60)
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/{MODEL_KEY}_category_and_time.png", dpi=150); plt.show()

## 12. Input-variation and robustness experiments (5 variations)
V1 prompt wording - V2 increasing input complexity - V3 ambiguous instructions - V4 increasing mathematical complexity - V5 different structured-output formats.
For every item the record stores the input, expected behaviour, actual output, evaluation result, inference time and an automatic observation.

In [ ]:
rob_recs = ca.run_samples(ROB, gen, expects_think=EXPECTS_THINK, tag="robust")
ca.save_json(f"{RESULTS_DIR}/{MODEL_KEY}_robustness.json", rob_recs)
by_var = {}
for r in rob_recs:
    d = by_var.setdefault(r["variation"], {"n": 0, "correct": 0, "sec": []}); d["n"] += 1; d["correct"] += int(r["eval"]["correct"]); d["sec"].append(r["seconds"])
ROB_SUMMARY = {k: {"n": v["n"], "correct": v["correct"], "accuracy": v["correct"] / v["n"], "mean_seconds": sum(v["sec"]) / len(v["sec"])} for k, v in by_var.items()}
display(pd.DataFrame(ROB_SUMMARY).T)
pd.DataFrame([{"id": r["id"], "variation": r["variation"], "pass": r["eval"]["correct"], "sec": round(r["seconds"], 1), "observation": r["observation"][:90]} for r in rob_recs])

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3.6))
ax.bar(list(ROB_SUMMARY), [v["accuracy"] for v in ROB_SUMMARY.values()], color="#5b8c5a"); ax.set_ylim(0, 1); ax.set_ylabel("accuracy"); ax.set_title(f"{MODEL_KEY}: robustness accuracy by variation"); ax.tick_params(axis="x", rotation=25)
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/{MODEL_KEY}_robustness.png", dpi=150); plt.show()

### Consistency check (same prompt, 3 seeds)
DeepSeek uses sampling (T=0.6, top-p 0.95) as the README recommends (greedy decoding raises repetition, paper §3). Different seeds can therefore give different reasoning traces and sometimes different answers; the table shows how many *distinct final answers* appeared across 3 seeds.

In [ ]:
CONS = [ca.consistency_test(gen, s, seeds=(0, 1, 2), expects_think=EXPECTS_THINK) for s in DEV if s["id"] in ("MR01", "LR02", "MS02")]
ca.save_json(f"{RESULTS_DIR}/{MODEL_KEY}_consistency.json", CONS)
for r in CONS: print(r["id"], "distinct answers:", r["n_distinct_answers"], "| all correct:", r["all_correct"], [o["extracted"] for o in r["runs"]])

## 13. Failure and limitation analysis
**Part A - OBSERVED failures** (automatically collected from this run; nothing is invented. If fewer than five failures occurred, the table is simply shorter.)
**Part B - THEORETICAL limitations** (from the paper/model card; *not* observed in this run) is a separate list below.

In [ ]:
FAILS = ca.build_failure_table([(r, ET[r["id"]]) for r in dev_recs + test_recs + rob_recs])
# context-length limitation evidence is appended after the resource experiments (section 14)
print(len(FAILS), "observed failures so far")
fdf = pd.DataFrame(FAILS)
fdf[["id", "category", "failure_type", "input", "actual_output"]].head(40) if len(FAILS) else print("No failures observed in dev/test/robustness runs.")

**Part B - THEORETICAL limitations (not observed in this run; from the paper / README)**
- Prompt sensitivity: *few-shot prompting consistently degrades* R1 performance; zero-shot with a clear output format is recommended (paper §5).
- Language mixing outside Chinese/English; may reason in English for other languages (paper §5).
- Weaker than DeepSeek-V3 on function calling, multi-turn, role-play and JSON output (paper §5, about the full R1 - the distilled 7B is expected to be no better).
- May skip the thinking pattern (`<think>\n\n</think>`) unless forced (README).
- Long reasoning traces: high latency and memory per answer; endless repetition at low/zero temperature (README).
- The distilled models received SFT only - the paper notes RL on them would likely help further and was not done.
- Safety: full R1 refuses some queries after safety RL (paper); the distilled checkpoint's safety behaviour is not characterised in the paper.

## 14. Resource experiments (same checkpoint, controlled variables)
(a) **input length** and (b) **generation length** on the primary (4-bit on T4) model, then (c) **precision variants** reloading the *same* checkpoint: 4-bit vs 8-bit, plus fp16 **only if the GPU has >= 22 GB** (otherwise recorded as SKIPPED with the reason). 4 dev prompts keep the 7B reruns affordable.

In [ ]:
MAX_CTX = LOAD_INFO.get("max_position_embeddings")
LENGTHS = [256, 1024, 2048, 4096, 6000]
RES_INPUT = ca.resource_input_length_experiment(model, tok, gen, LENGTHS, max_ctx=MAX_CTX, max_new_tokens=768, expects_think=True)
RES_GEN = ca.resource_generation_length_experiment(gen, [64, 256, 512])
SUBSET_IDS = ["GK02", "MR01", "LR02", "JS01"]
SUBSET = [s for s in DEV if s["id"] in SUBSET_IDS] if not QUICK_RUN else DEV[:2]
ca.release(model); model = None
gpu_gb = ENV.get("gpu_total_gb") or 0
precs = ["4bit", "8bit"] + (["bf16" if ENV.get("bf16_native") else "fp16"] if gpu_gb >= 22 else [])
RES_GEN_KW = dict(GEN_KW, max_new_tokens=min(MAX_NEW_TOKENS, 768))
RES_PRECISION = ca.resource_precision_experiment(MODEL_ID, HF_TOKEN, precs, SUBSET, RES_GEN_KW, expects_think=True, force_think_prefix=True)
if gpu_gb < 22:
    RES_PRECISION.append({"precision": "fp16/bf16", "status": f"SKIPPED: needs ~16+ GB for weights alone; this GPU has {gpu_gb} GB"})
ca.save_json(f"{RESULTS_DIR}/{MODEL_KEY}_resource.json", {"input_length": RES_INPUT, "generation_length": RES_GEN, "precision": RES_PRECISION})
display(pd.DataFrame(RES_INPUT)); display(pd.DataFrame(RES_GEN))
pd.DataFrame([{"precision": r["precision"], "status": r["status"][:60], **({"load_s": r["load"]["model_load_s"], "peak_gb": r["peak_mem_gb"], "acc": r["metrics"]["accuracy"], "mean_s": r["metrics"]["time_mean_s"], "tok_per_s": r["metrics"]["tokens_per_sec_aggregate"]} if r["status"] == "OK" else {})} for r in RES_PRECISION])

In [ ]:
# Append observed context-length failures to the failure table
for row in RES_INPUT:
    if row.get("status") == "OK" and not row["found_needle"]:
        FAILS.append({"id": "LEN%d" % row["target_tokens"], "category": "context_length", "input": f"~{row['actual_input_tokens']}-token log with one secret code in the middle",
                      "expected_behaviour": "retrieve the code K7-4921", "actual_output": row["answer_head"], "failure_type": "Context-length limitation (needle not retrieved)",
                      "explanation": "Observed in the input-length experiment at the stated prompt length.", "observed": True})
ca.save_json(f"{RESULTS_DIR}/{MODEL_KEY}_failure_cases.json", FAILS)
print(len(FAILS), "observed failure cases saved")

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(15, 3.8))
ok = [r for r in RES_PRECISION if r["status"] == "OK"]
if ok:
    ax[0].bar([r["precision"] for r in ok], [r["load"]["model_load_s"] for r in ok], color="#3b6ea5"); ax[0].set_title("load time (s, cached weights)")
    ax[1].bar([r["precision"] for r in ok], [r["peak_mem_gb"] or 0 for r in ok], color="#a55b3b"); ax[1].set_title("peak GPU memory (GB)")
    ax[2].bar([r["precision"] for r in ok], [r["metrics"]["accuracy"] for r in ok], color="#5b8c5a"); ax[2].set_ylim(0, 1); ax[2].set_title("accuracy on resource subset")
plt.tight_layout(); plt.savefig(f"{CHART_DIR}/{MODEL_KEY}_resource_precision.png", dpi=150); plt.show()
il = [r for r in RES_INPUT if r.get("status") == "OK"]
if il:
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
    ax[0].plot([r["actual_input_tokens"] for r in il], [r["seconds"] for r in il], "o-"); ax[0].set_xlabel("input tokens"); ax[0].set_ylabel("seconds"); ax[0].set_title("latency vs input length")
    ax[1].plot([r["actual_input_tokens"] for r in il], [r["peak_mem_gb"] for r in il], "o-", color="#a55b3b"); ax[1].set_xlabel("input tokens"); ax[1].set_ylabel("peak GB"); ax[1].set_title("peak memory vs input length")
    plt.tight_layout(); plt.savefig(f"{CHART_DIR}/{MODEL_KEY}_resource_input_length.png", dpi=150); plt.show()

## 15. Save everything (consumed by `G19_G20_Comparison.ipynb`)

In [ ]:
import datetime
RESULTS = {
    "meta": {"model_key": MODEL_KEY, "checkpoint": MODEL_ID, "display_name": PRIMARY_NAME, "precision": PRECISION, "quick_run": QUICK_RUN,
             "run_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(), "env": ENV, "generation_config": GEN_KW, "expects_think": EXPECTS_THINK,
             "force_think_prefix": FORCE_THINK_PREFIX, "load": LOAD_INFO, "fallback_attempts": tried, "dataset_version": DS["version"],
             "n_dev": len(DEV), "n_test": len(TEST), "n_robustness": len(ROB)},
    "dev": {"metrics": dev_metrics, "records": dev_recs}, "test": {"metrics": test_metrics, "records": test_recs},
    "robustness": {"summary": ROB_SUMMARY, "records": rob_recs}, "consistency": CONS, "failures": FAILS,
    "resource": {"precision": RES_PRECISION, "input_length": RES_INPUT, "generation_length": RES_GEN},
}
ca.save_json(f"{RESULTS_DIR}/{MODEL_KEY}_results.json", RESULTS)
for name, recs in (("dev", dev_recs), ("test", test_recs), ("robustness", rob_recs)):
    pd.DataFrame([{**{k: r[k] for k in ("id", "category", "prompt", "final_output", "seconds", "new_tokens", "tokens_per_sec", "hit_max_tokens")},
                   "correct": r["eval"]["correct"], "extracted": r["eval"]["extracted"], "observation": r["observation"]} for r in recs]).to_csv(f"{RESULTS_DIR}/{MODEL_KEY}_{name}_predictions.csv", index=False)
with open(f"{RESULTS_DIR}/{MODEL_KEY}_execution_log.txt", "w") as f:
    f.write(json.dumps(RESULTS["meta"], indent=1, default=str))
print("saved ->", sorted(os.listdir(RESULTS_DIR)))
ZIP = shutil.make_archive(f"/content/{MODEL_KEY}_results", "zip", RESULTS_DIR)
try:
    from google.colab import files; files.download(ZIP)
except Exception as e:
    print("download manually:", ZIP)

## 16. Final observations and conclusion

In [ ]:
from IPython.display import Markdown
t = test_metrics; d = dev_metrics
Markdown(f"""**Auto-generated summary of THIS run (verify before quoting):**

- Checkpoint `{MODEL_ID}` loaded at **{PRECISION}** on **{ENV.get('gpu')}**; load time {LOAD_INFO['model_load_s']:.1f}s (+{LOAD_INFO['download_s']:.1f}s download).
- Dev accuracy {d['accuracy']:.2f} ({d['n_correct']}/{d['n']}); held-out test accuracy {t['accuracy']:.2f} ({t['n_correct']}/{t['n']}), 95% Wilson CI {t['accuracy_ci95'][0]:.2f}-{t['accuracy_ci95'][1]:.2f}.
- Mean / median test latency {t['time_mean_s']:.1f}s / {t['time_median_s']:.1f}s; aggregate throughput {t['tokens_per_sec_aggregate']:.1f} tokens/s; peak GPU memory {t['peak_gpu_mem_gb_max']} GB.
- Responses cut off by the token limit: dev {d['n_hit_max_tokens']}, test {t['n_hit_max_tokens']}; unfinished reasoning traces: dev {d['n_unfinished_thinking']}, test {t['n_unfinished_thinking']}.
- Observed failures logged: {len(FAILS)} (see `{MODEL_KEY}_failure_cases.json`).
""")

### Observations (fill in after your run - these are *your* findings, not pre-written)
- How often did the model finish thinking within `MAX_NEW_TOKENS`? (`n_unfinished_thinking`, `n_hit_max_tokens`.) How much of the latency is reasoning vs final answer?
- Which categories gained from long reasoning (maths, multi-step, logic) and which suffered (format-constrained outputs, short factual answers)?
- Did 4-bit vs 8-bit (and fp16 if run) change accuracy / answers on the subset? Memory / speed trade-off?
- Consistency across seeds.

### Conclusion
*(Write 5-8 sentences from the tables above. Remember: this is a distilled 7B SFT model, not the full R1; do not generalise beyond n=30 items.)*